In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Load Data

In [3]:
df = pd.read_csv("transaction_data.csv", sep=",", encoding="utf-8")

### Check Data. Preprocessing

In [ ]:
r, c = df.shape
print(f"Rows: {r}, columns: {c}")

In [ ]:
df.head(3)

In [ ]:
df.describe(include="all").T

In [ ]:
df.dtypes

In [ ]:
print(f'Unfilled rows:\n{df.isna().sum()}')
print(f'\nDuplicated rows: {df.duplicated().sum()}')

In [ ]:
df.transaction = df.transaction.fillna('cancelled')

In [ ]:
df.minute = df.minute.fillna(0)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.transaction.unique()

In [ ]:
df['transaction'] = df['transaction'].str.lower()

df['transaction'] = df['transaction'].replace({'Successfull': 'successfull',
                       'succesfull': 'successfull',
                       'successful': 'successfull',
                       'canceled': 'cancelled',
                       'cencelled': 'cancelled',
                       'errror': 'error',
                       'eror': 'error'})

In [ ]:
transactions_vc = df.transaction.value_counts().sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(8, 6))

sns.barplot(x=transactions_vc.index,
            y=transactions_vc.values)

plt.title('Number of transations')
plt.xlabel('Type')
plt.ylabel('Count')

plt.show()

In [ ]:
print(f'Error transactions: {transactions_vc["error"]}')

In [ ]:
transations_by_users = df[df['transaction'] == 'successfull'].name.value_counts().reset_index()

In [ ]:

transations_by_users.head(3)

In [ ]:
sns.displot(data=transations_by_users, x='count')

plt.xlabel('Size')
plt.show()

In [ ]:
sns.displot(data=transations_by_users, x='count', bins=15)

plt.xlabel('Size')
plt.show()

In [ ]:
transations_by_users.describe()

In [ ]:
df = pd.read_csv('5_transaction_data_updated.csv', parse_dates=['date'])

In [ ]:
df.dtypes

In [ ]:
transactions_users_by_time = df.groupby(['name', 'minute']) \
    .agg({'date': 'count'}) \
    .rename(columns={'date': 'transactions'}) \
    .reset_index()

transactions_users_by_time.head()

In [ ]:
users_by_time_df = transactions_users_by_time.pivot(columns='name', values='transactions', index='minute')
users_by_time_df = users_by_time_df.fillna(0)
users_by_time_df.head()

In [ ]:
minute_trans = users_by_time_df.sum(axis=1)

In [ ]:
minute_trans.plot.bar(figsize=(10, 8))
plt.show()

In [ ]:
df.head()

In [ ]:
df['true_minute'] = df.date.dt.minute + df.date.dt.hour * 60

In [ ]:
df.head()

In [ ]:
sns.displot(data=df, x='true_minute', kde=True)

plt.xlabel('The number of minutes elapsed since the beginning of the day')
plt.ylabel('Count')

plt.show()

### Other datasets

In [28]:
tm_sales_1 = pd.read_csv('subsid/tm_sales_1.csv', sep=';', parse_dates=['ACT_DTTM'])
tm_sales_2 = pd.read_csv('subsid/tm_sales_2.csv', sep=';', parse_dates=['ACT_DTTM'])
tm_sales_3 = pd.read_csv('subsid/tm_sales_3.csv', sep=';', parse_dates=['ACT_DTTM'])
tm_sales_4 = pd.read_csv('subsid/tm_sales_4.csv', sep=';', parse_dates=['ACT_DTTM'])
tm_sales_5 = pd.read_csv('subsid/tm_sales_5.csv', sep=';', parse_dates=['ACT_DTTM'])

subs_logs = pd.read_csv('subsid/subscription_logs.csv', sep=';', parse_dates=['START_DTTM', 'END_DTTM'])

In [33]:
tm_sales = pd.concat([tm_sales_1, tm_sales_2, tm_sales_3, tm_sales_4, tm_sales_5], ignore_index=False)

In [38]:
tm_sales.shape

(2500, 4)

In [39]:
tm_sales = tm_sales.dropna(subset=['SUBS_ID'])

In [40]:
tm_sales.shape

(2425, 4)

In [47]:
tm_sales.loc[~tm_sales['SUBS_ID'].str.startswith('id'),
    'SUBS_ID'] = 'id' + tm_sales.loc[~tm_sales['SUBS_ID'].str.startswith('id'), 'SUBS_ID']

In [53]:
subs_logs = subs_logs.merge(right=tm_sales, on=['SUBS_ID'])

In [19]:
# 5 minutes = 300 seconds -> sales more than 5 minutes (300 seconds)
five_minutes = 300

In [54]:
success_subs_df = subs_logs[
    (subs_logs['END_DTTM'] - subs_logs['START_DTTM']).dt.seconds > five_minutes
]

In [55]:
success_subs_df.head

<bound method NDFrame.head of      SUBS_ID          START_DTTM            END_DTTM PROD_ID_x  \
0     id9871 2020-03-02 06:38:00 2020-03-14 04:38:00      P004   
1     id4451 2020-03-02 09:05:00 2020-03-10 17:05:00      P003   
2     id5981 2020-03-02 04:58:00 2020-03-11 20:58:00      P001   
3     id8657 2020-03-02 00:30:00 2020-03-14 12:30:00      P004   
5     id2610 2020-03-02 06:32:00 2020-03-11 05:32:00      P003   
...      ...                 ...                 ...       ...   
1285  id3140 2020-03-02 01:27:00 2020-03-03 00:27:00      P003   
1286  id4233 2020-03-02 13:24:00 2020-03-07 11:24:00      P001   
1287  id9260 2020-03-02 02:41:00 2020-03-14 08:41:00      P003   
1288  id9647 2020-03-02 07:00:00 2020-03-11 13:00:00      P001   
1289  id5074 2020-03-02 13:32:00 2020-03-11 04:32:00      P004   

            ACT_DTTM_x  FILIAL_ID_x PROD_ID_y        ACT_DTTM_y  FILIAL_ID_y  \
0     2020-03-02 13:24            1      P004  2020-03-02 13:24            1   
1     02-03-2020 

In [26]:
success_subs_df.merge(right=temp, on='SUBS_ID')

,SUBS_ID,START_DTTM,END_DTTM,PROD_ID,ACT_DTTM,FILIAL_ID
0,id3870,2020-03-02 08:03:00,2020-03-06 15:03:00,P004,02-03-2020 01:49,8
1,id5215,2020-03-02 10:52:00,2020-03-06 23:52:00,P002,2020-03-02 06:38,2
2,id9313,2020-03-02 01:08:00,2020-03-13 11:08:00,P004,2020-03-02 06:48,2
3,id1659,2020-03-02 10:23:00,2020-03-08 09:23:00,P002,02-03-2020 00:40,1
4,id4728,2020-03-02 04:09:00,2020-03-14 14:09:00,P001,02-03-2020 04:13,2
...,...,...,...,...,...,...
95,id5833,2020-03-02 03:01:00,2020-03-10 05:01:00,P001,2020-03-02 00:06,6
96,id9341,2020-03-02 07:45:00,2020-03-09 03:45:00,P001,2020-03-02 15:28,7
97,id1981,2020-03-02 01:38:00,2020-03-04 23:38:00,P001,2020-03-02 04:15,3
98,id2465,2020-03-02 03:14:00,2020-03-02 21:14:00,P003,02-03-2020 01:05,5


In [58]:
success_subs_df.SUBS_ID.sort_values().values

<StringArray>
['id1003', 'id1051', 'id1070', 'id1076', 'id1080', 'id1111', 'id1116',
 'id1138', 'id1168', 'id1173',
 ...
 'id9897', 'id9897', 'id9897', 'id9897', 'id9904', 'id9919', 'id9946',
 'id9948', 'id9948', 'id9955']
Length: 1143, dtype: str

In [ ]:
success_subs_df.to_csv('')